
# VCLI-G Methods (Geometry-Coupled Visual Cognitive Load Index)

VCLI-G is a research-grade scoring system designed to measure how much perceptual and cognitive work an image demands — not how “pretty” it is, not how similar it is to training data, and not how well it matches a text prompt.
It couples perceptual arrest (how hard an image is to resolve or “let go of”) with geometric signals extracted directly from the composition.

It’s meant to answer a single, difficult question:

How much does this image make me think?

The system is built around four measurable geometric signals:

G1 — Centroid Wander (attention instability): how much the visual center shifts as you scan the image.

G2 — Void Topology (figure/ground ambiguity): how space and absence are organized, and whether they resist resolution.

G3 — Curvature Torque (formal tension): how much structural pressure or opposition is present in directional flow.

G4 — Occlusion Entropy (proxy) (depth/ambiguity): how near/far relationships and overlaps create perceptual uncertainty.

What this is not

❌ A style or aesthetics score — it does not reward “beautiful” images.

❌ A similarity or CLIP-like model — it does not care how close an image is to its text prompt.

❌ A quality ranking — a low score can mean “intentionally simple” or “fully resolved,” not “bad.”

❌ A finished metric — this is a scaffold, designed to be clear, auditable, and extendable.

❌ It is not a composition or aesthetics score. It does not judge balance, color harmony, polish, or “good design.”

Score	Cognitive Load	Viewer Behavior	Meaning

VCLI-G returns a composite score on a 0.0 – 5.0 scale:

0.0 – 1.0 → Low cognitive load – The image resolves quickly. It’s stable, obvious, or visually quiet.

1.0 – 3.0 → Moderate load – There’s some structural complexity, but resolution is straightforward.

3.0 – 4.0 → High load – The image resists closure. Tension, contradiction, or ambiguity hold your attention.

4.0 – 5.0 → Very high load – The image “won’t let go.” It’s perceptually sticky, demands re-entry, or unfolds across multiple viewings.

Treat VCLI-G as a delay index — a measure of how long an image resists resolution, not how “good” it is.

Compare scores to understand viewer effort or attention gravity across images or runs.

Use it alongside structural tools to reveal where and why perceptual strain occurs — but don’t confuse high scores with aesthetic merit

⚠️ **Important:**
- A high VCLI-G score does not mean "better." It means "more cognitively demanding."
- A high SCI score does not mean "better." It means "more organized."
- In some artistic or design contexts, that’s desirable. In others, a low-load image may be the right result.
- In some contexts, low VCLI-G + high SCI is ideal (branding, UI).
- In others, high VCLI-G + low SCI is valuable (experimental art, emergent process).
- The scores describe what's happening, not whether it's good.

Why this matters for AI

Modern reward models and diffusion pipelines are tuned to favor images that resolve quickly — centered, balanced, clear, and “understandable.”
That’s useful for general purpose generation, but it suppresses the very qualities artists often care about: tension, delay, contradiction, recursive looking.

VCLI-G offers a way to quantify and reintroduce those qualities into evaluation loops.
Used alongside CLIP scores, aesthetic ratings, or model logits, it helps researchers and artists:

Detect when a model is playing it safe visually.

Explore regions of the latent space that produce consequential work.

Benchmark changes in perceptual engagement across model versions or prompt strategies.

## Two Metrics, One System

VCLI-G is now paired with **SCI (Structural Coherence Index)** to form a 2D perceptual space:

**VCLI-G (0.0 – 5.0)**: How much cognitive work does the image demand?
**SCI (0.0 – 5.0)**: How organized is that work?

They're independent axes that together describe different kinds of visual complexity:

**High VCLI-G, High SCI** → Earned tension (Cézanne, deliberate ambiguity)
**High VCLI-G, Low SCI** → Chaotic complexity (glitch, noise, productive accidents)
**Low VCLI-G, High SCI** → Resolved clarity (Vermeer, intentional simplicity)
**Low VCLI-G, Low SCI** → Default simplicity (gradient + centered object)

⚠️ **Neither axis is a quality score.** Low SCI doesn't mean "bad" — it means emergent/process-driven (like Pollock). High SCI doesn't mean "good" — it means systematic/composed. Context determines which is appropriate.

## SCI Score Bands

**0.0 – 1.5** → Entropic — High local variance, scattered edges, inconsistent detail. Complexity is emergent or random.

**1.5 – 3.0** → Mixed — Some organizational structure, but irregular. Parts may be coherent while others drift.

**3.0 – 4.0** → Coherent — Clear structural logic. Edges align, detail is consistent, rhythm is present.

**4.0 – 5.0** → Highly organized — Systematic, deliberate decisions visible throughout. Strong internal consistency.

**SCI measures organization, not intention.** An algorithmic pattern can score high SCI. A deliberate "messy" sketch can score low SCI. Both are valid depending on context.

## Why SCI Matters Alongside VCLI-G

VCLI-G alone can't distinguish:
- Intentional ambiguity from incoherent noise
- Earned tension from accidental chaos  
- Deliberate simplicity from lazy defaults

**SCI provides context.** It reveals whether high cognitive load comes from:
- **Organized friction** (high SCI) → deliberate formal tension, layered decisions
- **Entropic friction** (low SCI) → emergent complexity, process-driven, experimental

This lets you:
- Filter AI outputs for "playing it safe" (low VCLI-G, high SCI)
- Identify consequential work (high VCLI-G, high SCI)
- Detect when models collapse into noise (high VCLI-G, low SCI)
- Find productive accidents (low SCI but interesting)

**Use case examples:**
- **UI/UX design** → Filter for low VCLI-G, high SCI (clear, organized)
- **Gallery curation** → Hunt for high VCLI-G, high SCI (earned complexity)
- **Experimental work** → Embrace high VCLI-G, low SCI (productive chaos)
- **Model diagnostics** → Track when outputs drift into low SCI (overfitting to texture)

In [ ]:

# If running on Colab, uncomment to install dependencies:
# !pip -q install opencv-python-headless scikit-image shapely networkx matplotlib


In [ ]:

import numpy as np
import cv2
import matplotlib.pyplot as plt

from skimage import filters, feature, measure, morphology, segmentation, color
from skimage.measure import label, regionprops, euler_number, find_contours
from skimage.morphology import remove_small_holes, remove_small_objects
from scipy.ndimage import gaussian_filter, distance_transform_edt
from scipy.signal import savgol_filter

import networkx as nx

# Utility plotting
def show(img, title=None, cmap='gray'):
    plt.figure(figsize=(6,6))
    if img.ndim == 2:
        plt.imshow(img, cmap=cmap)
    else:
        import cv2 as _cv2
        plt.imshow(_cv2.cvtColor(img, _cv2.COLOR_BGR2RGB))
    if title: plt.title(title)
    plt.axis('off')
    plt.show()


In [ ]:

def load_image(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(f"Could not read image at {path}")
    return img

def to_gray(img):
    if img.ndim == 3:
        g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    else:
        g = img.copy()
    return g.astype(np.float32) / 255.0

def saliency_map(gray):
    gx = filters.sobel_h(gray)
    gy = filters.sobel_v(gray)
    return np.hypot(gx, gy)

def saliency_pyramid(gray, sigmas=(2,4,8,16)):
    maps = []
    for s in sigmas:
        g = gaussian_filter(gray, sigma=s)
        maps.append(saliency_map(g))
    return maps


In [ ]:

def weighted_centroid(smap):
    sm = smap + 1e-8
    h, w = sm.shape
    y, x = np.mgrid[0:h, 0:w]
    m = sm.sum()
    cx = (sm * x).sum() / m
    cy = (sm * y).sum() / m
    return np.array([cx, cy])

def path_length(points):
    if len(points) < 2:
        return 0.0
    diffs = np.diff(points, axis=0)
    return np.linalg.norm(diffs, axis=1).sum()

def polyline_turn_curvature(points):
    """
    Mean absolute turning angle (radians) per segment.
    Stable for small N.
    """
    if len(points) < 3:
        return 0.0
    pts = np.asarray(points, dtype=float)
    v = np.diff(pts, axis=0)               # segment vectors
    # remove near-zero segments to avoid noise
    lens = np.linalg.norm(v, axis=1) + 1e-8
    v = v[lens > 1e-6]
    if len(v) < 2:
        return 0.0
    # normalize and compute turning angles
    v = v / (np.linalg.norm(v, axis=1, keepdims=True) + 1e-8)
    dots = np.clip((v[:-1] * v[1:]).sum(axis=1), -1.0, 1.0)
    angs = np.arccos(dots)                 # [0..pi]
    return float(np.mean(np.abs(angs)))

def G1_centroid_wander(gray, sigmas=(1.5, 3, 6, 12, 24)):
    """
    Multiscale centroids + path metrics.
    - More scales for better shape.
    - K from discrete turning angles (robust at low N).
    """
    pyr = saliency_pyramid(gray, sigmas=sigmas)
    pts = np.array([weighted_centroid(p) for p in pyr], dtype=float)

    # path length (pixels)
    diffs = np.diff(pts, axis=0)
    L = float(np.linalg.norm(diffs, axis=1).sum()) if len(pts) > 1 else 0.0

    # curvature (radians/turn)
    K = polyline_turn_curvature(pts)
    return {"points": pts, "path_len": L, "path_curv": K}


In [ ]:
# --- Void Topology v2 (clean) ---
# Brightness-thresholded voids + robust metrics (no wild Euler spikes)

import numpy as np
import cv2
from skimage.measure import label, regionprops, euler_number
from skimage.morphology import (
    remove_small_holes, remove_small_objects, binary_opening, binary_closing, disk
)
from scipy.ndimage import distance_transform_edt

def _edges_for_viz(gray):
    g8  = (gray * 255).astype(np.uint8)
    med = np.median(g8)
    lo  = int(max(0, (1.0 - 0.66) * med))
    hi  = int(min(255, (1.0 + 0.66) * med))
    return cv2.Canny(g8, lo, hi) > 0

def _void_mask_v2(gray, ksize=31, C=-0.02, min_size=256):
    """
    Build a void mask by treating locally *brighter* regions as 'voids' (holes in mass),
    then clean with morphology to suppress texture speckle.
    """
    g = (gray - gray.min()) / (gray.max() - gray.min() + 1e-8)
    g_blur = cv2.GaussianBlur(g, (0, 0), 3.0)

    th = cv2.adaptiveThreshold(
        (g_blur * 255).astype(np.uint8),
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        ksize | 1,
        int(C * 255),
    )
    voids = th.astype(bool)
    voids = binary_opening(voids, disk(2))
    voids = binary_closing(voids, disk(3))
    voids = remove_small_objects(voids, min_size=min_size)
    voids = remove_small_holes(voids, area_threshold=min_size)
    return voids

def void_metrics(gray):
    """
    Returns:
      V         : number of detected voids (bright regions)
      chi       : bounded mass-fragmentation proxy (negative, capped)
      AR        : aspect ratio of the largest void's bbox
      cut_depth : max interior distance of largest void (pixels)
      edges     : Canny edges (for visualization only)
      voids     : boolean mask of voids
    """
    voids = _void_mask_v2(gray)
    edges = _edges_for_viz(gray)

    lab = label(voids)
    V   = int(lab.max())

    # Largest-void features
    if V == 0:
        AR = 1.0
        cut_depth = 0.0
    else:
        props = sorted(regionprops(lab), key=lambda r: r.area, reverse=True)
        p0    = props[0]
        (minr, minc, maxr, maxc) = p0.bbox
        H = max(1, maxr - minr)
        W = max(1, maxc - minc)
        AR = max(H, W) / float(min(H, W))
        cut_depth = float(distance_transform_edt(lab == p0.label).max())

    # Bounded fragmentation proxy for 'chi' (keeps API, avoids huge magnitudes)
    # Interpret as: more mass fragments (non-void islands) → more negative.
    mass = ~voids
    n_islands = int(label(mass).max())
    chi = float(-min(128, n_islands))  # cap to keep scores stable

    return {
        "V": V,
        "chi": chi,
        "AR": float(AR),
        "cut_depth": float(cut_depth),
        "edges": edges,
        "voids": voids,
    }

In [ ]:

def contour_curvature_stats(gray, level=None):
    g = gaussian_filter(gray, 1.0)
    if level is None:
        level = float(g.mean())
    contours = find_contours(g, level=level)
    if not contours:
        return {"k_var":0.0, "infl_density":0.0, "n":0}
    curvs = []
    infl = 0
    for c in contours:
        if len(c) < 9:
            continue
        x = c[:,1]; y = c[:,0]
        k = 9 if 9 < len(c) else (len(c)-1 if (len(c)-1)%2==1 else len(c)-2)
        xs = savgol_filter(x, k, 2, mode='interp')
        ys = savgol_filter(y, k, 2, mode='interp')
        dx = np.gradient(xs); dy = np.gradient(ys)
        ddx = np.gradient(dx); ddy = np.gradient(dy)
        denom = (dx*dx + dy*dy)**1.5 + 1e-8
        kappa = (dx*ddy - dy*ddx) / denom  # signed curvature
        curvs.append(kappa)
        infl += int(np.sum(np.sign(kappa[:-1]) != np.sign(kappa[1:])))
    if not curvs:
        return {"k_var":0.0, "infl_density":0.0, "n":0}
    curvs = np.concatenate(curvs)
    k_var = float(np.var(curvs))
    infl_density = float(infl / max(1, curvs.size))
    return {"k_var": k_var, "infl_density": infl_density, "n": int(curvs.size)}


In [ ]:

def orientation_entropy(gray, bins=36):
    gx = filters.sobel_h(gray)
    gy = filters.sobel_v(gray)
    mag = np.hypot(gx, gy) + 1e-8
    ang = (np.arctan2(gy, gx) + np.pi) % np.pi
    hist, _ = np.histogram(ang, bins=bins, range=(0,np.pi), weights=mag)
    p = hist / (hist.sum() + 1e-8)
    H = -np.sum(p * np.log2(p + 1e-12))

    # Scaling to ensure output stays in 0-10 range
    H_max = np.log2(bins)
    H_normalized = (H / H_max) * 12.0  # Changed from 10.0 to 8.0
    return float(H_normalized)

def G4_occlusion_entropy_proxy(gray):
    # Proxy: higher orientation entropy ~ more competing structure (potential near/far ambiguity)
    H = orientation_entropy(gray, bins=36)
    return {"H": H}


In [ ]:
# --- T-junction occlusion DAG (compact reference) ---

import cv2, numpy as np, networkx as nx
from skimage import feature, segmentation, morphology
from skimage.filters import sobel

def tjunctions(gray):
    g = (gray*255).astype(np.uint8)
    e = cv2.Canny(g, 50, 150)  # tweakable
    # Harris corners to preselect interest points
    h = cv2.cornerHarris(g, 2, 3, 0.04)
    ys, xs = np.where(h > 0.01*h.max())

    # local orientation via structure tensor
    gx, gy = np.gradient(gray)
    ang = (np.arctan2(gy, gx) + np.pi) % np.pi   # [0,pi)

    T = []
    for y, x in zip(ys, xs):
        y0, y1 = max(0,y-3), min(gray.shape[0], y+4)
        x0, x1 = max(0,x-3), min(gray.shape[1], x+4)
        win = e[y0:y1, x0:x1]
        if win.sum() < 12:  # not enough strokes
            continue
        # crude: find two dominant angle modes
        aw = ang[y0:y1, x0:x1][win>0]
        if aw.size < 10:
            continue
        hist, bins = np.histogram(aw, bins=18, range=(0,np.pi))
        modes = np.argsort(hist)[-2:]
        a0 = 0.5*(bins[modes[0]] + bins[modes[0]+1])
        a1 = 0.5*(bins[modes[1]] + bins[modes[1]+1])
        # large angle difference ≈ T (vs X/Y)
        if abs(np.sin(a0 - a1)) < 0.6:  # reject near-parallel
            continue
        # Heuristic occluder: orientation with the shorter visible segment → "bar"
        # We approximate by counting pixels aligned with each mode.
        n0 = hist[modes[0]]; n1 = hist[modes[1]]
        bar = 0 if n0 < n1 else 1
        T.append((y, x, a0, a1, bar))
    return T, e

def regions_from_edges(gray, edges):
    # watershed on gradient -> labeled regions
    elev = sobel(gray)
    m = morphology.remove_small_holes(~edges.astype(bool), 64)
    lab = segmentation.watershed(elev, markers=None, mask=m)
    return lab

def occlusion_graph(gray):
    T, e = tjunctions(gray)
    lab = regions_from_edges(gray, e)
    G = nx.DiGraph()
    G.add_nodes_from(np.unique(lab)[1:])  # skip 0
    h, w = gray.shape
    for (y, x, a0, a1, bar) in T:
        # sample normals to pick near regions
        y0, x0 = int(np.clip(y,0,h-1)), int(np.clip(x,0,w-1))
        nbh = lab[max(0,y0-1):min(h,y0+2), max(0,x0-1):min(w,x0+2)]
        regs = [r for r in np.unique(nbh) if r != 0]
        if len(regs) < 2:
            continue
        # very rough: bar region occludes the others at the junction
        occluder = regs[bar % len(regs)]
        for r in regs:
            if r != occluder:
                G.add_edge(occluder, r)
    return G

def occlusion_entropy_dag(G: nx.DiGraph) -> float:
    if not G.number_of_nodes():
        return 0.0
    indeg = np.array([d for _, d in G.in_degree()], float)
    outdeg = np.array([d for _, d in G.out_degree()], float)

    def ent(v):
        p = v / (v.sum() + 1e-8)
        return float(-np.sum(p * np.log2(p + 1e-12))) if (p > 0).any() else 0.0

    H_in, H_out = ent(indeg + 1e-8), ent(outdeg + 1e-8)
    H = H_in + H_out
    # normalize to a friendly band (~0..6). You can calibrate with your corpus.
    return float(H)

def G4_occlusion_entropy_true(gray):
    G = occlusion_graph(gray)
    H = occlusion_entropy_dag(G)
    return {"H": H}

# ---- G4 switch (proxy vs true DAG) ----
USE_TRUE_G4 = False   # set False to fall back to proxy quickly

def G4_extract(gray):
    if USE_TRUE_G4:
        return G4_occlusion_entropy_true(gray)
    else:
        return G4_occlusion_entropy_proxy(gray)

## G5 — Structural Coherence Index (SCI)

**What it measures**: Whether complexity is organized or entropic.

**Four sub-signals**:
1. **Regional Consistency** (30% weight) — Do different areas of the image have similar structural properties?
2. **Edge Alignment** (30% weight) — Do edges follow dominant angles, or scatter randomly?
3. **Scale Consistency** (25% weight) — Does detail density stay proportional across zoom levels?
4. **Rhythm/Repetition** (15% weight) — Are there systematic patterns via autocorrelation?

**Returns**: SCI composite (0–5) plus four sub-metrics.

**High SCI** → Edges align, regions are consistent, detail is proportional, rhythm is present.
**Low SCI** → High local variance, scattered orientations, detail spikes unpredictably.

**Not measured**: Whether the organization is "good composition." SCI detects systematic structure, not aesthetic merit. A perfectly organized gradient scores high; a masterful Pollock scores low. Both are valid.

In [ ]:
# --- Structural Coherence Index (SCI) ---
import numpy as np
import cv2
from scipy.signal import correlate2d
from skimage.filters import sobel

def structural_coherence_index(gray):
    """
    Measures whether complexity is organized vs. entropic.
    Returns dict with sub-metrics and composite SCI score (0-5).

    Low SCI = chaotic, entropic, scattered
    High SCI = organized, coherent, systematic
    """
    h, w = gray.shape

    # 1. Regional Consistency - divide into 4 quadrants, compare variance
    qh, qw = h // 2, w // 2
    quadrants = [
        gray[0:qh, 0:qw],
        gray[0:qh, qw:w],
        gray[qh:h, 0:qw],
        gray[qh:h, qw:w]
    ]

    # Edge density per quadrant
    q_edge_density = []
    for q in quadrants:
        edges = sobel(q)
        q_edge_density.append(float(edges.mean()))

    # Low variance across quadrants = consistent structure
    regional_consistency = 1.0 / (1.0 + np.var(q_edge_density))

    # 2. Edge Alignment - measure angular entropy of edges
    gx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    mag = np.hypot(gx, gy)
    angles = np.arctan2(gy, gx)

    # Weight by magnitude and compute histogram
    angle_bins = 36
    hist, _ = np.histogram(angles[mag > np.percentile(mag, 25)],
                          bins=angle_bins, range=(-np.pi, np.pi))

    # Entropy of angle distribution (low = aligned, high = scattered)
    p = hist / (hist.sum() + 1e-8)
    angle_entropy = -np.sum(p * np.log2(p + 1e-12))
    max_entropy = np.log2(angle_bins)
    angle_alignment = 1.0 - (angle_entropy / max_entropy)  # Invert so high = aligned

    # 3. Scale Consistency - detail density across pyramid levels
    pyramid_levels = []
    current = gray.copy()
    for _ in range(4):
        edges = sobel(current)
        pyramid_levels.append(float(edges.mean()))
        if min(current.shape) < 32:
            break
        current = cv2.pyrDown(current)

    if len(pyramid_levels) > 1:
        scale_consistency = 1.0 / (1.0 + np.std(pyramid_levels) / (np.mean(pyramid_levels) + 1e-8))
    else:
        scale_consistency = 1.0

    # 4. Rhythm/Repetition - autocorrelation strength
    # Downsample for speed
    small = cv2.resize(gray, (min(128, w), min(128, h)))
    small = (small - small.mean()) / (small.std() + 1e-8)

    # Compute autocorrelation
    autocorr = correlate2d(small, small, mode='same')
    autocorr = autocorr / autocorr.max()

    # Find peaks excluding center
    center_y, center_x = np.array(autocorr.shape) // 2
    mask = np.ones_like(autocorr, dtype=bool)
    mask[center_y-5:center_y+5, center_x-5:center_x+5] = False

    peaks = autocorr[mask] > 0.3  # Threshold for significant peaks
    rhythm_strength = float(np.sum(peaks)) / 100.0  # Normalize
    rhythm_strength = min(1.0, rhythm_strength)

    # Composite SCI score
    # Weights: regional consistency (0.3), angle alignment (0.3),
    #          scale consistency (0.25), rhythm (0.15)
    sci_raw = (0.30 * regional_consistency +
               0.30 * angle_alignment +
               0.25 * scale_consistency +
               0.15 * rhythm_strength)

    # Map to 0-5 scale (already in 0-1 range, so just scale)
    sci_score = float(np.clip(sci_raw * 5.0, 0, 5))

    return {
        "SCI": sci_score,
        "regional_consistency": float(regional_consistency),
        "angle_alignment": float(angle_alignment),
        "scale_consistency": float(scale_consistency),
        "rhythm_strength": float(rhythm_strength),
    }

In [ ]:
# --- Running stats & REFS (required by strict scorer) ---
class RunningStats:
    def __init__(self):
        self.values = []
    def update(self, v):
        self.values.append(float(v))
    def z(self, v):
        import numpy as np
        vals = np.array(self.values) if self.values else np.array([v])
        mu = float(vals.mean())
        sd = float(vals.std() + 1e-8)
        return (float(v) - mu) / sd

# Global refs (empty is OK; you can seed priors later)
REFS = {
    "G1_L":   RunningStats(),
    "G1_K":   RunningStats(),
    "G2_V":   RunningStats(),
    "G2_chi": RunningStats(),
    "G2_AR":  RunningStats(),
    "G2_cut": RunningStats(),
    "G3_kvar":RunningStats(),
    "G3_infl":RunningStats(),
    "G4_H":   RunningStats(),
}

Choose a profile in toggle

ai_conservative,— Understates spiky cases, illumination and lowers rewards for AI "false" complexity with void/texture filling.
Slightly down-weights voids (w2) and tightens the z-clip, so giant single-void images (e.g., strong spotlight/triangle) don’t dominate. Use for editorial/UX contexts where false highs are costly and you prefer “prove it” behavior.

physical_neutral — Baseline calibration.
Balanced weights, mild z-clip, standard squash. Good for most images; tracks the raw geometry signals without favoring any one (voids, torque, wander, entropy). Use when you want comparable scores across mixed sets.

physical_balanced_plus — Allows earned lift, physcical muted tonal work, but still sane.
Modestly up-weights voids and loosens the z-clip a bit so clean, intentional figure/ground tension can register. Keeps the same squash, no global bias. Use for curated/art sets where you want to surface images with deliberate arrest without rewarding chaos.

In [ ]:
# --- VCLI-G Profiles (sits on top of compute_vclig_strict) ---
import numpy as np

VCLIG_PROFILES = {
  # "Prove it" - skeptical of cheap complexity
  "ai_conservative": dict(
    weights=dict(w1=0.28, w2=0.14, w3=0.32, w4=0.18, w5=0.08),
    # ↑ Down-weight voids (w2), up-weight wander/torque (w1/w3)
    z_clip=3.0,      # Tighter ceiling - less tolerance for spikes
    squash=2.8,      # Slightly compressed output
    bias=-+0.2        # Modest (-0.2 ≈ -0.15 points on 0-5 scale)
  ),

  # "Trust the math" - balanced, no agenda
  "physical_neutral": dict(
    weights=dict(w1=0.25, w2=0.20, w3=0.27, w4=0.20, w5=0.08),
    # ↑ Balanced across all four signals
    z_clip=3.5,      # Standard tolerance
    squash=2.5,      # Standard spread
    bias=+1.0         # Adjustment
  ),

  # "Hear the tension" - credit for earned complexity
  "physical_balanced_plus": dict(
    weights=dict(w1=0.24, w2=0.22, w3=0.29, w4=0.19, w5=0.06),
    # ↑ Modest boost to voids (w2) and torque (w3)
    z_clip=4.0,      # More permissive ceiling
    squash=2.5,      # Same spread as neutral
    bias=+1.5        # Modest optimism (+0.2 ≈ +0.15 points)
  ),
}

# Active profile toggle
VCLIG_ACTIVE_PROFILE = "ai_conservative"

def set_profile(name: str):
    global VCLIG_ACTIVE_PROFILE
    assert name in VCLIG_PROFILES, f"Unknown profile '{name}'. Options: {list(VCLIG_PROFILES)}"
    VCLIG_ACTIVE_PROFILE = name
    print("Active profile ->", VCLIG_ACTIVE_PROFILE)

def score_with_profile(image_path, profile=None):
    name = profile or VCLIG_ACTIVE_PROFILE
    cfg  = VCLIG_PROFILES[name]
    s, f, z = compute_vclig_strict(
        image_path,
        weights=cfg["weights"],
        z_clip=cfg["z_clip"],
        squash=cfg["squash"],
    )
    s = float(np.clip(s + cfg["bias"], 0, 5))
    return s, f, z, name

In [ ]:

def compute_vclig(image_path, sigmas=(2,4,8,16), delta_rec=0.0, visualize=False):
    img = load_image(image_path)
    gray = to_gray(img)

    g1 = G1_centroid_wander(gray, sigmas=sigmas)
    g2 = void_metrics(gray)
    g3 = contour_curvature_stats(gray)
    g4 = G4_extract(gray)

    feats = {
        "G1_L": g1["path_len"],
        "G1_K": g1["path_curv"],
        "G2_V": g2["V"],
        "G2_chi": g2["chi"],
        "G2_AR": g2["AR"],
        "G2_cut": g2["cut_depth"],
        "G3_kvar": g3["k_var"],
        "G3_infl": g3["infl_density"],
        "G4_H": g4["H"],
    }
    score, zinfo = vclig_score_coldstart_safe(
        feats, weights=None, squash=2.5, z_clip=None, delta_rec=delta_rec
    )

    if visualize:
        pts = g1["points"]
        fig, ax = plt.subplots(figsize=(5,5))
        ax.imshow(gray, cmap='gray')
        ax.plot(pts[:,0], pts[:,1], '-o')
        ax.set_title("G1: Centroid path (multiscale)")
        ax.axis('off')
        plt.show()

        show(g2["voids"], "G2: Void map")
        show(feature.canny(gray), "Edges")
        print("G3 curvature variance:", g3["k_var"], " inflection density:", g3["infl_density"])
        print("G4 orientation entropy:", g4["H"])

    return score, feats, zinfo


In [ ]:
# --- Cold-start fallback & robust normalization patch ---

import numpy as np

# 1) Replace Euler number with bounded "hole ratio"
def _bounded_holes(voids):
    # holes = number of 0-islands inside the largest connected 1-region
    # simple bound: min(128, count); normalize by area later if desired
    from skimage.measure import label
    lab = label(voids)
    return min(128, int(lab.max()))

# override in void_metrics post-process
# --- VCLI-G: side-effect–free extraction + clean scoring (drop-in cell) ---
# Requires the earlier helpers already in your notebook:
# load_image, to_gray, G1_centroid_wander, void_metrics,
# contour_curvature_stats, G4_occlusion_entropy_proxy, and the REFS dict.

import numpy as np

# normalize heavy-tailed features
def _normalize_feats(feats: dict) -> dict:
    f = dict(feats)
    if "G2_chi" in f:
        f["G2_chi"] = -min(128.0, abs(float(f["G2_chi"])))  # bounded, monotone
    f["G2_cut"]  = np.log1p(max(0.0, f["G2_cut"]))
    f["G3_kvar"] = np.log1p(max(0.0, f["G3_kvar"]))
    return f

# 1) Pure feature extraction — NO updates to REFS (no side effects)
def extract_features_noupdate(image_path, sigmas=(2,4,8,16)):
    """Pure features. NO REFS mutation."""
    img  = load_image(image_path)
    gray = to_gray(img)

    g1 = G1_centroid_wander(gray, sigmas=sigmas)
    g2 = void_metrics(gray)
    g3 = contour_curvature_stats(gray)
    g4 = G4_extract(gray)
    sci_data = structural_coherence_index(gray)  # ADD THIS LINE

    return {
        "G1_L": float(g1["path_len"]),
        "G1_K": float(g1["path_curv"]),
        "G2_V": int(g2["V"]),
        "G2_chi": float(g2["chi"]),
        "G2_AR": float(g2["AR"]),
        "G2_cut": float(g2["cut_depth"]),
        "G3_kvar": float(g3["k_var"]),
        "G3_infl": float(g3["infl_density"]),
        "G4_H": float(g4["H"]),
        # ADD THESE LINES:
        "SCI": float(sci_data["SCI"]),
        "SCI_regional": float(sci_data["regional_consistency"]),
        "SCI_angle": float(sci_data["angle_alignment"]),
        "SCI_scale": float(sci_data["scale_consistency"]),
        "SCI_rhythm": float(sci_data["rhythm_strength"]),
    }

# 2) Robust z-score for cold start (falls back when refs are empty or σ≈0)
def _robust_z(x, series):
    arr = np.array(series) if len(series) else np.array([x])
    med = float(np.median(arr))
    mad = float(np.median(np.abs(arr - med))) + 1e-6
    return (float(x) - med) / (1.4826 * mad)

def _z_or_robust(key, val):
    ref = REFS[key].values
    return _robust_z(val, ref) if (len(ref) < 3 or np.std(ref) < 1e-6) else REFS[key].z(val)

# 3) Cold-start–safe scorer (0–5 band)
def vclig_score_coldstart_safe(features, *, weights=None, squash=2.5, z_clip=None, delta_rec=0.0):
    """Return (score[0..5], {'z1','z2','z3','z4','raw'}) without mutating REFS."""
    if weights is None:
        weights = dict(w1=0.25, w2=0.20, w3=0.25, w4=0.20, w5=0.10)

    f = _normalize_feats(features)

    z1 = _z_or_robust("G1_L", f["G1_L"]) + 0.5 * _z_or_robust("G1_K", f["G1_K"])
    z2_raw = (_z_or_robust("G2_V",  f["G2_V"]) +
          0.75 * _z_or_robust("G2_cut", f["G2_cut"]) +
          0.5  * _z_or_robust("G2_AR",  f["G2_AR"]) +
          0.5  * _z_or_robust("G2_chi", f["G2_chi"]))
    z2 = z2_raw / 2.5  # Normalize by sum of weights to keep scale consistent
    z3 = (_z_or_robust("G3_kvar", f["G3_kvar"]) +
          _z_or_robust("G3_infl", f["G3_infl"]))
    z4 = _z_or_robust("G4_H", f["G4_H"])

    if z_clip is not None:
        clip = lambda v: max(-z_clip, min(z_clip, v))
        z1, z2, z3, z4 = map(clip, (z1, z2, z3, z4))

    raw = (weights["w1"]*z1 + weights["w2"]*z2 +
           weights["w3"]*z3 + weights["w4"]*z4 + weights["w5"]*delta_rec)

    score = 2.5 + np.tanh(raw / float(squash)) * 2.5
    return float(np.clip(score, 0, 5)), {"z1": float(z1), "z2": float(z2), "z3": float(z3), "z4": float(z4), "raw": float(raw)}

# 4) Strict API — no REFS mutation anywhere during extraction
def compute_vclig_strict(image_path, *, sigmas=(2,4,8,16), weights=None, squash=2.5, z_clip=None, delta_rec=0.0):
    feats = extract_features_noupdate(image_path, sigmas=sigmas)
    score, zinfo = vclig_score_coldstart_safe(feats, weights=weights, squash=squash, z_clip=z_clip, delta_rec=delta_rec)
    return score, feats, zinfo


In [ ]:
# Clear any previous leakage
for r in REFS.values(): r.values.clear()

# Optional: seed priors (edit if you like)
seed = {
  "G1_L": [1, 5, 12, 25, 45, 75],
  "G1_K": [0.0, 0.05, 0.20, 0.45, 0.80, 1.3],

  "G2_V": [1, 8, 25, 60, 140, 280],
  "G2_chi": [-1, -2, -3, -5, -10, -20],
  "G2_AR": [1.0, 1.3, 2.5, 5.0, 9.0, 16.0],
  "G2_cut": [1.5, 3.5, 5.5, 7.5, 9.5, 11],

  "G3_kvar": [0.2, 3, 50, 400, 3000, 18000],
  "G3_infl": [0.005, 0.03, 0.07, 0.11, 0.15, 0.18],

  "G4_H": [2, 5, 7.5, 9.5, 11, 12.5],
}
for k, vals in seed.items():
    for v in vals: REFS[k].update(v)

In [ ]:

# @title Reset: clear /content/images
import shutil, os
IMG_DIR = "/content/images"
if os.path.exists(IMG_DIR):
    shutil.rmtree(IMG_DIR)
os.makedirs(IMG_DIR, exist_ok=True)
print("Reset:", IMG_DIR)



## Upload Images (Colab)
Use this cell in **Google Colab** to upload one or more local images.  
The next cell shows how to run VCLI-G on the uploaded files and save a CSV.


In [ ]:

# --- Colab Upload Cell ---
# Run this in Google Colab to upload images from your computer.
# After running, use the 'uploaded_paths' list below.
try:
    from google.colab import files  # type: ignore
    up = files.upload()  # opens file picker
    uploaded_paths = list(up.keys())
    print("Uploaded:", uploaded_paths)
except Exception as e:
    print("Note: This cell is intended for Google Colab.")
    print("Error/Info:", e)
    uploaded_paths = []


In [ ]:
# --- Full results table (features + z's) with profile, plus CSV ---

import pandas as pd
import numpy as np

ALL_COLS = [
    "path", "profile", "VCLI_G",
    "G1_L", "G1_K",
    "G2_V", "G2_chi", "G2_AR", "G2_cut",
    "G3_kvar", "G3_infl",
    "G4_H",
    "SCI", "SCI_regional", "SCI_angle", "SCI_scale", "SCI_rhythm",
    "z_z1", "z_z2", "z_z3", "z_z4", "z_raw",
]

def batch_vclig_with_profile(paths, profile=None):
    """Strict scoring for a list of paths, honoring the active (or provided) profile."""
    name = profile or VCLIG_ACTIVE_PROFILE
    rows = []
    for p in paths:
        try:
            s, f, z, _ = score_with_profile(p, name)
            rows.append({
                "path": p,
                "profile": name,
                "VCLI_G": round(float(s), 3),
                **{k: float(v) for k, v in f.items()},
                "z_z1": float(z["z1"]),
                "z_z2": float(z["z2"]),
                "z_z3": float(z["z3"]),
                "z_z4": float(z["z4"]),
                "z_raw": float(z["raw"]),
            })
        except Exception as e:
            rows.append({"path": p, "profile": name, "error": str(e)})
    return rows

def results_table(paths, *, profile=None, sort_by="VCLI_G", ascending=False, csv_path=None):
    data = batch_vclig_with_profile(paths, profile=profile)
    df = pd.DataFrame(data)

    # ensure all columns exist and order them
    for c in ALL_COLS:
        if c not in df.columns:
            df[c] = np.nan
    df = df[ALL_COLS + [c for c in df.columns if c not in ALL_COLS]]

    # round numerics a bit for readability
    for c in df.columns:
        if pd.api.types.is_numeric_dtype(df[c]):
            df[c] = df[c].astype(float).round(4)

    if sort_by in df.columns:
        df = df.sort_values(sort_by, ascending=ascending).reset_index(drop=True)

    display(df)
    if csv_path:
        df.to_csv(csv_path, index=False)
        print("Saved to", csv_path)
    return df

**SET PROFILE CELL BELOW, LINE 6**

In [ ]:
POOL = (
    uploaded_paths
    if ("uploaded_paths" in globals() and uploaded_paths)
    else globals().get("image_paths", [])
)
set_profile("ai_conservative")  # or ai_conservative, physical_neutral, physical_balanced_plus
df = results_table(POOL, csv_path="/content/VCLI_G_results.csv")

# choose a profile once if you want
# set_profile ai_conservative, physical_neutral, physical_balanced_plus

In [ ]:
# 2) Disable truncation for the session
import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 2000)
df.head()
display(df)

In [ ]:
# @title 1. Signal Correlation Matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Show correlation between raw features and z-scores
feature_cols = ['G1_L', 'G1_K', 'G2_V', 'G2_chi', 'G2_AR', 'G2_cut',
                'G3_kvar', 'G3_infl', 'G4_H']
z_cols = ['z_z1', 'z_z2', 'z_z3', 'z_z4']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Raw feature correlations
sns.heatmap(df[feature_cols].corr(), annot=True, fmt='.2f',
            cmap='coolwarm', center=0, ax=ax1, square=True)
ax1.set_title('Raw Feature Correlations', fontsize=13, weight='bold')

# Z-score correlations
sns.heatmap(df[z_cols].corr(), annot=True, fmt='.2f',
            cmap='coolwarm', center=0, ax=ax2, square=True)
ax2.set_title('Z-Score Correlations', fontsize=13, weight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# @title 2. Signal Contribution Chart (Top N Images)
import matplotlib.pyplot as plt
import numpy as np

TOP_N = 40  # Change this to see more/fewer images

# Get top scoring images
df_top = df.nlargest(TOP_N, 'VCLI_G').copy()

# Calculate weighted contributions using active profile
cfg = VCLIG_PROFILES[VCLIG_ACTIVE_PROFILE]
w = cfg['weights']

df_top['c1'] = df_top['z_z1'] * w['w1']
df_top['c2'] = df_top['z_z2'] * w['w2']
df_top['c3'] = df_top['z_z3'] * w['w3']
df_top['c4'] = df_top['z_z4'] * w['w4']
df_top['c5'] = w['w5']  # delta_rec contribution

# Plot stacked bar
fig, ax = plt.subplots(figsize=(14, 7))
df_top[['c1', 'c2', 'c3', 'c4', 'c5']].plot(
    kind='bar', stacked=True, ax=ax,
    color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A', '#E0E0E0'],
    edgecolor='black', linewidth=0.7
)

# Format x-axis labels (shorten paths)
labels = [p.split('/')[-1][:25] + '...' if len(p.split('/')[-1]) > 25
          else p.split('/')[-1] for p in df_top['path']]
ax.set_xticklabels(labels, rotation=45, ha='right')

ax.set_ylabel('Weighted Contribution to Raw Score', fontsize=11)
ax.set_xlabel('Image', fontsize=11)
ax.set_title(f'Signal Contributions to VCLI-G (Top {TOP_N}, Profile: {VCLIG_ACTIVE_PROFILE})',
             fontsize=13, weight='bold', pad=15)
ax.legend(['z1 (Wander)', 'z2 (Void)', 'z3 (Torque)', 'z4 (Occlusion)', 'Δ_rec'],
          loc='upper left', fontsize=10)
ax.axhline(0, color='black', linewidth=0.8, alpha=0.7)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nTop {TOP_N} images by VCLI-G score:")
for idx, row in df_top.iterrows():
    print(f"  {row['path'].split('/')[-1]}: {row['VCLI_G']:.3f}")

In [ ]:
# @title 3. VCLI-G vs Individual Signals (Scatter + Regression)
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(13, 11))
axes = axes.flatten()

z_cols = ['z_z1', 'z_z2', 'z_z3', 'z_z4']
labels = ['z1 (Centroid Wander)', 'z2 (Void Topology)',
          'z3 (Curvature Torque)', 'z4 (Occlusion Entropy)']
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']

for i, (zcol, label, color) in enumerate(zip(z_cols, labels, colors)):
    ax = axes[i]
    ax.scatter(df[zcol], df['VCLI_G'], alpha=0.6, s=60,
               edgecolors='black', linewidth=0.5, color=color)

    # LABEL ONLY THE 4 EXTREME POINTS
    extremes = [
        df[zcol].idxmax(),  # Highest on x-axis
        df[zcol].idxmin(),  # Lowest on x-axis
        df['VCLI_G'].idxmax(),  # Highest on y-axis
        df['VCLI_G'].idxmin(),  # Lowest on y-axis
    ]
    for idx in set(extremes):  # set() removes duplicates
        row = df.loc[idx]
        img_name = row['path'].split('/')[-1][:15]
        ax.annotate(img_name,
                   (row[zcol], row['VCLI_G']),
                   xytext=(8, 8), textcoords='offset points',
                   fontsize=8, alpha=0.8,
                   bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.6),
                   arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0.2'))

    ax.set_xlabel(label, fontsize=11, weight='bold')
    ax.set_ylabel('VCLI-G Score', fontsize=11)
    ax.grid(True, alpha=0.3, linestyle='--')

    # Add regression line
    if len(df) > 1:
        z = np.polyfit(df[zcol], df['VCLI_G'], 1)
        p = np.poly1d(z)
        x_line = np.linspace(df[zcol].min(), df[zcol].max(), 100)
        ax.plot(x_line, p(x_line), "r--", alpha=0.8, linewidth=2.5)

        # Correlation coefficient
        corr = df[[zcol, 'VCLI_G']].corr().iloc[0, 1]
        ax.text(0.05, 0.95, f'r = {corr:.3f}', transform=ax.transAxes,
                fontsize=11, weight='bold', verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))

plt.suptitle(f'VCLI-G vs Individual Signals (Profile: {VCLIG_ACTIVE_PROFILE})',
             fontsize=14, weight='bold', y=0.998)
plt.tight_layout()
plt.show()

# Print correlation summary
print("\nCorrelation Summary (signal → VCLI-G):")
for zcol, label in zip(z_cols, labels):
    corr = df[[zcol, 'VCLI_G']].corr().iloc[0, 1]
    print(f"  {label}: r = {corr:.3f}")

In [ ]:
# @title 4. Score and Signal Distributions
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# VCLI-G distribution
ax = axes[0, 0]
ax.hist(df['VCLI_G'], bins=20, edgecolor='black', alpha=0.75, color='steelblue')
ax.axvline(df['VCLI_G'].median(), color='red', linestyle='--',
          linewidth=2.5, label=f"Median: {df['VCLI_G'].median():.2f}")
ax.axvline(df['VCLI_G'].mean(), color='orange', linestyle='--',
          linewidth=2.5, label=f"Mean: {df['VCLI_G'].mean():.2f}")
ax.set_xlabel('VCLI-G Score', fontsize=11, weight='bold')
ax.set_ylabel('Count', fontsize=11, weight='bold')
ax.set_title('VCLI-G Distribution', fontsize=12, weight='bold')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

# z-score distributions
z_cols = ['z_z1', 'z_z2', 'z_z3', 'z_z4']
labels = ['z1 (Wander)', 'z2 (Void)', 'z3 (Torque)', 'z4 (Occlusion)']
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']

for i, (zcol, z_label, color) in enumerate(zip(z_cols, labels, colors)):
    if i < 2:  # FIXED: was i < 3
        ax = axes[0, i + 1]
    else:
        ax = axes[1, i - 2]  # FIXED: was i - 3

    ax.hist(df[zcol], bins=20, edgecolor='black', alpha=0.75, color=color)
    ax.axvline(0, color='black', linestyle='-', linewidth=1.5, alpha=0.7)
    ax.axvline(df[zcol].mean(), color='darkred', linestyle='--',
              linewidth=2, alpha=0.8)
    ax.set_xlabel(z_label, fontsize=10, weight='bold')
    ax.set_ylabel('Count', fontsize=10, weight='bold')
    ax.set_title(f'{z_label} Distribution', fontsize=11, weight='bold')
    ax.grid(axis='y', alpha=0.3)
    ax.text(0.98, 0.97, f'μ={df[zcol].mean():.2f}\nσ={df[zcol].std():.2f}',
            transform=ax.transAxes, fontsize=9, verticalalignment='top',
            horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

# Raw score distribution
ax = axes[1, 1]  # FIXED: was axes[1, 1] but now needs to be consistent
ax.hist(df['z_raw'], bins=20, edgecolor='black', alpha=0.75, color='mediumpurple')
ax.axvline(df['z_raw'].median(), color='red', linestyle='--', linewidth=2.5)
ax.axvline(0, color='black', linestyle='-', linewidth=1.5, alpha=0.7)
ax.set_xlabel('Raw Composite (pre-squash)', fontsize=10, weight='bold')
ax.set_ylabel('Count', fontsize=10, weight='bold')
ax.set_title('Raw Score Distribution', fontsize=11, weight='bold')
ax.grid(axis='y', alpha=0.3)

# Summary stats box
ax = axes[1, 2]
ax.axis('off')
summary_text = f"""
DISTRIBUTION SUMMARY
{'='*30}

VCLI-G:
  Mean:   {df['VCLI_G'].mean():.3f}
  Median: {df['VCLI_G'].median():.3f}
  Std:    {df['VCLI_G'].std():.3f}
  Range:  [{df['VCLI_G'].min():.3f}, {df['VCLI_G'].max():.3f}]

Signal Means:
  z1: {df['z_z1'].mean():.3f}
  z2: {df['z_z2'].mean():.3f}
  z3: {df['z_z3'].mean():.3f}
  z4: {df['z_z4'].mean():.3f}

Profile: {VCLIG_ACTIVE_PROFILE}
N = {len(df)} images
"""
ax.text(0.1, 0.5, summary_text, fontsize=10, family='monospace',
        verticalalignment='center',
        bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.3))

plt.suptitle('Distribution Analysis', fontsize=15, weight='bold', y=0.998)
plt.tight_layout()
plt.show()

In [ ]:
# @title 5. Feature Space Projection (PCA)
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA

feature_cols = ['G1_L', 'G1_K', 'G2_V', 'G2_chi', 'G2_AR', 'G2_cut',
                'G3_kvar', 'G3_infl', 'G4_H']

# Prepare data
X = df[feature_cols].fillna(0).values

# Fit PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

# Plot
fig, ax = plt.subplots(figsize=(11, 9))
scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1],
                    c=df['VCLI_G'], cmap='viridis',
                    s=120, edgecolors='black', linewidth=0.7,
                    alpha=0.8)

# Add labels for extreme points
for idx in [df['VCLI_G'].idxmax(), df['VCLI_G'].idxmin()]:
    label = df.iloc[idx]['path'].split('/')[-1][:15]
    ax.annotate(label, (X_pca[idx, 0], X_pca[idx, 1]),
               xytext=(10, 10), textcoords='offset points',
               fontsize=9, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7),
               arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0.3'))

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)',
             fontsize=12, weight='bold')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)',
             fontsize=12, weight='bold')
ax.set_title('Feature Space Projection (PCA)\n9D features → 2D',
            fontsize=14, weight='bold', pad=15)
ax.grid(True, alpha=0.3, linestyle='--')

cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('VCLI-G Score', rotation=270, labelpad=25, fontsize=11, weight='bold')

plt.tight_layout()
plt.show()

# Print component breakdown
print("\nPCA Component Loadings:")
print("="*50)
components_df = pd.DataFrame(
    pca.components_.T,
    columns=['PC1', 'PC2'],
    index=feature_cols
)
print(components_df.round(3).to_string())
print(f"\nTotal variance explained: {pca.explained_variance_ratio_.sum():.1%}")

## Reading the 2D Space (VCLI-G × SCI)

When analyzing images, consider their coordinates:

**Quadrant interpretations**:

| SCI ↓ / VCLI-G → | **Low (0-2.5)** | **High (2.5-5.0)** |
|------------------|-----------------|-------------------|
| **High (2.5-5)** | Resolved, clear, organized | Earned tension, deliberate complexity |
| **Low (0-2.5)**  | Default simple, unstructured | Chaotic, emergent, experimental |

**Movement across the space**:
- An artist refining a sketch might move from *low SCI → high SCI* (organizing chaos)
- A model "playing it safe" clusters in *low VCLI-G, high SCI*
- Overfitting to texture noise pushes toward *high VCLI-G, low SCI*
- Master works often live in *high VCLI-G, high SCI* (but not always!)

**The space is descriptive, not prescriptive.** Where an image should live depends on context, medium, and intent.

In [ ]:
# @title 6. VCLI-G vs SCI (2D Perceptual Space)
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(12, 10))

# Scatter plot
scatter = ax.scatter(df['SCI'], df['VCLI_G'],
                    s=120, alpha=0.7,
                    c=df['VCLI_G'], cmap='viridis',
                    edgecolors='black', linewidth=0.8)

# Add quadrant lines
ax.axhline(2.5, color='gray', linestyle='--', linewidth=1.5, alpha=0.5)
ax.axvline(2.5, color='gray', linestyle='--', linewidth=1.5, alpha=0.5)

# Label quadrants
ax.text(0.5, 4.5, 'Chaotic\nComplexity', fontsize=11, ha='center',
        style='italic', alpha=0.6, weight='bold')
ax.text(4.5, 4.5, 'Earned\nTension', fontsize=11, ha='center',
        style='italic', alpha=0.6, weight='bold')
ax.text(0.5, 0.5, 'Default\nSimple', fontsize=11, ha='center',
        style='italic', alpha=0.6, weight='bold')
ax.text(4.5, 0.5, 'Resolved\nClarity', fontsize=11, ha='center',
        style='italic', alpha=0.6, weight='bold')

# Annotate extremes
for idx in [df['VCLI_G'].idxmax(), df['VCLI_G'].idxmin(),
            df['SCI'].idxmax(), df['SCI'].idxmin()]:
    row = df.iloc[idx]
    img_label = row['path'].split('/')[-1][:12]
    ax.annotate(img_label, (row['SCI'], row['VCLI_G']),
               xytext=(8, 8), textcoords='offset points',
               fontsize=8, alpha=0.8,
               bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.6),
               arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0.2'))

ax.set_xlabel('SCI (Structural Coherence Index)', fontsize=13, weight='bold')
ax.set_ylabel('VCLI-G (Visual Cognitive Load)', fontsize=13, weight='bold')
ax.set_title('Perceptual Space: Complexity vs. Coherence',
            fontsize=14, weight='bold', pad=15)
ax.set_xlim(-0.2, 5.2)
ax.set_ylim(-0.2, 5.2)
ax.grid(True, alpha=0.3, linestyle=':')

# Add colorbar
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('VCLI-G', rotation=270, labelpad=20, fontsize=11)

plt.tight_layout()
plt.show()

# Print coordinate summary
print("\nPerceptual Space Coordinates:")
print("="*60)
for idx, row in df.iterrows():
    img_name = row['path'].split('/')[-1][:30]
    print(f"{img_name:32s} | VCLI-G: {row['VCLI_G']:.2f} | SCI: {row['SCI']:.2f}")

In [ ]:
# @title 7. SCI Component Breakdown
import matplotlib.pyplot as plt

TOP_N = 50

df_top = df.nlargest(TOP_N, 'SCI').copy()

# Extract SCI components
components = ['SCI_regional', 'SCI_angle', 'SCI_scale', 'SCI_rhythm']
labels = ['Regional\nConsistency', 'Edge\nAlignment', 'Scale\nConsistency', 'Rhythm']

fig, ax = plt.subplots(figsize=(14, 7))

# Create stacked bar chart (each component contributes to total)
# Weight them according to the formula: 0.30, 0.30, 0.25, 0.15
df_top['c_regional'] = df_top['SCI_regional'] * 0.30 * 5
df_top['c_angle'] = df_top['SCI_angle'] * 0.30 * 5
df_top['c_scale'] = df_top['SCI_scale'] * 0.25 * 5
df_top['c_rhythm'] = df_top['SCI_rhythm'] * 0.15 * 5

df_top[['c_regional', 'c_angle', 'c_scale', 'c_rhythm']].plot(
    kind='bar', stacked=True, ax=ax,
    color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#95E1D3'],
    edgecolor='black', linewidth=0.7
)

# Format
labels_short = [p.split('/')[-1][:25] for p in df_top['path']]
ax.set_xticklabels(labels_short, rotation=45, ha='right')
ax.set_ylabel('Weighted Contribution to SCI', fontsize=11, weight='bold')
ax.set_xlabel('Image', fontsize=11, weight='bold')
ax.set_title(f'SCI Component Breakdown (Top {TOP_N} by SCI)',
             fontsize=13, weight='bold', pad=15)
ax.legend(labels, loc='upper left', fontsize=10)
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 5.5)

plt.tight_layout()
plt.show()

print(f"\nTop {TOP_N} images by SCI:")
for idx, row in df_top.iterrows():
    print(f"  {row['path'].split('/')[-1]:30s} SCI: {row['SCI']:.3f}")

In [ ]:
# @title 8. SCI Correlation with VCLI-G Signals
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(13, 11))
axes = axes.flatten()

z_cols = ['z_z1', 'z_z2', 'z_z3', 'z_z4']
labels = ['z1 (Centroid Wander)', 'z2 (Void Topology)',
          'z3 (Curvature Torque)', 'z4 (Occlusion Entropy)']
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']

for i, (zcol, z_label, color) in enumerate(zip(z_cols, labels, colors)):
    ax = axes[i]
    ax.scatter(df['SCI'], df[zcol], alpha=0.6, s=70,
               edgecolors='black', linewidth=0.5, color=color)

    # LABEL EXTREMES
    extremes = [df['SCI'].idxmax(), df['SCI'].idxmin(),
                df[zcol].idxmax(), df[zcol].idxmin()]
    for idx in set(extremes):
        row = df.loc[idx]
        img_name = row['path'].split('/')[-1][:12]
        ax.annotate(img_name, (row['SCI'], row[zcol]),
                   xytext=(8, 8), textcoords='offset points', fontsize=7,
                   bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.6),
                   arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0.2'))

    ax.set_xlabel('SCI (Structural Coherence)', fontsize=11, weight='bold')
    ax.set_ylabel(z_label, fontsize=11)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.axhline(0, color='black', linestyle='-', linewidth=1, alpha=0.4)
    ax.axvline(2.5, color='gray', linestyle='--', linewidth=1, alpha=0.4)

    # Regression line
    if len(df) > 1:
        z_fit = np.polyfit(df['SCI'], df[zcol], 1)
        p = np.poly1d(z_fit)
        x_line = np.linspace(df['SCI'].min(), df['SCI'].max(), 100)
        ax.plot(x_line, p(x_line), "r--", alpha=0.7, linewidth=2)

        # Correlation
        corr = df[['SCI', zcol]].corr().iloc[0, 1]
        ax.text(0.05, 0.95, f'r = {corr:.3f}', transform=ax.transAxes,
                fontsize=11, weight='bold', verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))

plt.suptitle('SCI vs VCLI-G Signals (Are coherence and load correlated?)',
             fontsize=14, weight='bold', y=0.998)
plt.tight_layout()
plt.show()

print("\nCorrelation Summary (SCI → VCLI-G signals):")
for zcol, z_label in zip(z_cols, labels):
    corr = df[['SCI', zcol]].corr().iloc[0, 1]
    print(f"  {z_label:30s}: r = {corr:6.3f}")
print(f"\n  SCI vs VCLI-G overall: r = {df[['SCI', 'VCLI_G']].corr().iloc[0, 1]:6.3f}")

In [ ]:
# @title 9. Component Variance Contribution Analysis
import numpy as np
import matplotlib.pyplot as plt

# Calculate variance contribution of each z-score to overall VCLI-G variance
signals = ['z_z1', 'z_z2', 'z_z3', 'z_z4']
signal_labels = ['z1 (Wander)', 'z2 (Void)', 'z3 (Torque)', 'z4 (Occlusion)']

# Variance of each signal
variances = [df[s].var() for s in signals]
total_var = sum(variances)

# Also compute weighted contribution (based on active profile)
cfg = VCLIG_PROFILES[VCLIG_ACTIVE_PROFILE]
weights = [cfg['weights']['w1'], cfg['weights']['w2'],
           cfg['weights']['w3'], cfg['weights']['w4']]
weighted_vars = [v * w for v, w in zip(variances, weights)]
total_weighted = sum(weighted_vars)

# Create comparison plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Raw variance contribution
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']
bars1 = ax1.bar(signal_labels, [v/total_var*100 for v in variances],
                color=colors, edgecolor='black', linewidth=1.2)
ax1.set_ylabel('% of Total Variance', fontsize=11, weight='bold')
ax1.set_title('Raw Variance Contribution', fontsize=12, weight='bold')
ax1.set_ylim(0, 100)
ax1.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar, val in zip(bars1, variances):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height + 1,
            f'{val/total_var:.1%}',
            ha='center', va='bottom', fontsize=10, weight='bold')

# Weighted variance contribution (profile-adjusted)
bars2 = ax2.bar(signal_labels, [v/total_weighted*100 for v in weighted_vars],
                color=colors, edgecolor='black', linewidth=1.2)
ax2.set_ylabel('% of Weighted Variance', fontsize=11, weight='bold')
ax2.set_title(f'Weighted Contribution (Profile: {VCLIG_ACTIVE_PROFILE})',
              fontsize=12, weight='bold')
ax2.set_ylim(0, 100)
ax2.grid(axis='y', alpha=0.3)

# Add value labels
for bar, val in zip(bars2, weighted_vars):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height + 1,
            f'{val/total_weighted:.1%}',
            ha='center', va='bottom', fontsize=10, weight='bold')

plt.tight_layout()
plt.show()

# Print detailed breakdown
print("\nVariance Contribution Analysis:")
print("="*60)
print(f"Profile: {VCLIG_ACTIVE_PROFILE}")
print(f"\n{'Signal':<20} {'Variance':<12} {'Weight':<10} {'Contribution':<12}")
print("-"*60)
for sig, label, var, w, wvar in zip(signals, signal_labels, variances, weights, weighted_vars):
    print(f"{label:<20} {var:>10.4f}  {w:>8.2f}  {wvar/total_weighted:>10.1%}")
print("-"*60)
print(f"{'Total':<20} {total_var:>10.4f}             {1.0:>10.1%}")

# Flag if any signal contributes <5%
low_contributors = [(label, wvar/total_weighted) for label, wvar in zip(signal_labels, weighted_vars)
                    if wvar/total_weighted < 0.05]
if low_contributors:
    print(f"\n⚠️  Signals contributing <5% (may not be pulling weight):")
    for label, contrib in low_contributors:
        print(f"   {label}: {contrib:.1%}")

In [ ]:
# @title 10. SCI Sub-Component Deep Dive
import matplotlib.pyplot as plt
import numpy as np

# SCI components
components = ['SCI_regional', 'SCI_angle', 'SCI_scale', 'SCI_rhythm']
labels = ['Regional\nConsistency', 'Edge\nAlignment', 'Scale\nConsistency', 'Rhythm\nStrength']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

# Individual histograms for each component
for i, (comp, label) in enumerate(zip(components, labels)):
    ax = axes[i]

    ax.hist(df[comp], bins=25, edgecolor='black', alpha=0.75, color='steelblue')
    ax.axvline(df[comp].mean(), color='red', linestyle='--', linewidth=2.5,
               label=f'μ={df[comp].mean():.3f}')
    ax.axvline(df[comp].median(), color='orange', linestyle='--', linewidth=2,
               label=f'median={df[comp].median():.3f}')

    ax.set_xlabel(label, fontsize=11, weight='bold')
    ax.set_ylabel('Count', fontsize=11)
    ax.set_title(f'{label} Distribution\nσ={df[comp].std():.3f}, range=[{df[comp].min():.3f}, {df[comp].max():.3f}]',
                 fontsize=11, weight='bold')
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

# Composite SCI histogram
ax = axes[4]
ax.hist(df['SCI'], bins=25, edgecolor='black', alpha=0.75, color='darkgreen')
ax.axvline(df['SCI'].mean(), color='red', linestyle='--', linewidth=2.5)
ax.axvline(2.5, color='gray', linestyle=':', linewidth=2, alpha=0.5, label='Threshold (2.5)')
ax.set_xlabel('SCI Composite', fontsize=11, weight='bold')
ax.set_ylabel('Count', fontsize=11)
ax.set_title(f'Overall SCI Distribution\nμ={df["SCI"].mean():.3f}, σ={df["SCI"].std():.3f}',
             fontsize=11, weight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Correlation between SCI components
ax = axes[5]
import seaborn as sns
corr = df[components].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax,
            square=True, cbar_kws={'label': 'Correlation'})
ax.set_title('SCI Component Correlations', fontsize=11, weight='bold')

plt.tight_layout()
plt.show()

# Print diagnostic summary
print("\n" + "="*70)
print("SCI COMPONENT ANALYSIS")
print("="*70)

print(f"\n{'Component':<20} {'Mean':<10} {'Std':<10} {'Min':<10} {'Max':<10} {'Range':<10}")
print("-"*70)
for comp, label in zip(components, labels):
    mean = df[comp].mean()
    std = df[comp].std()
    min_val = df[comp].min()
    max_val = df[comp].max()
    rng = max_val - min_val
    print(f"{label.replace(chr(10), ' '):<20} {mean:>8.3f}  {std:>8.3f}  {min_val:>8.3f}  {max_val:>8.3f}  {rng:>8.3f}")

print("-"*70)
print(f"{'SCI Composite':<20} {df['SCI'].mean():>8.3f}  {df['SCI'].std():>8.3f}  {df['SCI'].min():>8.3f}  {df['SCI'].max():>8.3f}  {df['SCI'].max()-df['SCI'].min():>8.3f}")

# Diagnosis
print("\nDIAGNOSIS:")
saturated = []
for comp, label in zip(components, labels):
    if df[comp].mean() > 0.95:
        saturated.append(f"{label.replace(chr(10), ' ')} (μ={df[comp].mean():.3f})")
    if df[comp].std() < 0.05:
        print(f"   ⚠️  {label.replace(chr(10), ' ')} has very low variance (σ={df[comp].std():.3f})")

if saturated:
    print(f"   🚨 Components saturating near 1.0: {', '.join(saturated)}")
    print(f"      → SCI may not be differentiating effectively")

if df['SCI'].std() < 0.15:
    print(f"   🚨 Overall SCI has low variance (σ={df['SCI'].std():.3f})")
    print(f"      → All images scoring as highly coherent")
    print(f"      → Either your dataset is genuinely uniform, or SCI needs recalibration")

In [ ]:
# @title 11. Signal Variance Contribution (Stacked View)
import matplotlib.pyplot as plt
import numpy as np

# For each image, show how much each z-score contributes to final score
cfg = VCLIG_PROFILES[VCLIG_ACTIVE_PROFILE]
weights = cfg['weights']

# Calculate weighted z contributions for top N images
TOP_N = 50
df_top = df.nlargest(TOP_N, 'VCLI_G').copy()

df_top['c1'] = df_top['z_z1'] * weights['w1']
df_top['c2'] = df_top['z_z2'] * weights['w2']
df_top['c3'] = df_top['z_z3'] * weights['w3']
df_top['c4'] = df_top['z_z4'] * weights['w4']

# Create waterfall-style chart
fig, ax = plt.subplots(figsize=(16, 8))

# Stacked bar with absolute contributions
df_top[['c1', 'c2', 'c3', 'c4']].plot(
    kind='bar', stacked=True, ax=ax,
    color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A'],
    edgecolor='black', linewidth=0.8, width=0.8
)

# Add VCLI-G score line
ax2 = ax.twinx()
ax2.plot(range(len(df_top)), df_top['VCLI_G'].values,
         'ko-', linewidth=2.5, markersize=8, label='Final VCLI-G')
ax2.set_ylabel('Final VCLI-G Score', fontsize=12, weight='bold', color='black')
ax2.set_ylim(0, 5)
ax2.legend(loc='upper right', fontsize=10)

# Format
labels = [p.split('/')[-1][:20] + ('...' if len(p.split('/')[-1]) > 20 else '')
          for p in df_top['path']]
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_ylabel('Weighted Contribution (Raw Score)', fontsize=12, weight='bold')
ax.set_xlabel('Image', fontsize=12, weight='bold')
ax.set_title(f'Signal Contributions to VCLI-G (Profile: {VCLIG_ACTIVE_PROFILE})',
             fontsize=14, weight='bold', pad=20)
ax.legend(['z1 (Wander)', 'z2 (Void)', 'z3 (Torque)', 'z4 (Occlusion)'],
          loc='upper left', fontsize=10, framealpha=0.9)
ax.axhline(0, color='black', linewidth=1, alpha=0.7)
ax.grid(axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

# Print contribution breakdown
print(f"\nSignal Contribution Breakdown (Top {TOP_N} images):")
print("="*90)
print(f"{'Image':<30} | {'z1×w1':<8} {'z2×w2':<8} {'z3×w3':<8} {'z4×w4':<8} | {'Raw':<7} | {'VCLI-G'}")
print("-"*90)
for _, row in df_top.iterrows():
    img_name = row['path'].split('/')[-1][:28]
    print(f"{img_name:<30} | {row['c1']:>6.2f}  {row['c2']:>6.2f}  {row['c3']:>6.2f}  {row['c4']:>6.2f}  | {row['z_raw']:>6.2f} | {row['VCLI_G']:>6.2f}")

Notes

This notebook is a reference implementation — clear and minimal for experimentation.

For production or research papers, replace the G4 proxy with a T-junction–driven occlusion DAG for more robust depth inference.

Because VCLI-G is intentionally geometry-coupled, it pairs well with structural metrics (e.g., LSI, OCF) to build richer evaluation frameworks.

⚠️ Interpreting Results — Common Pitfalls

Because VCLI-G measures perceptual load rather than polish or intent, it’s possible to misread a high score if you’re not careful. A few common edge cases:

False Positives (Fake Friction):
Some AI-generated images produce high scores simply because they are messy or incoherent — not because they contain deliberate tension. The system sees “delay,” but the viewer is really just confused. Always check whether friction feels intentional (driven by form, space, or symbolism) or accidental (caused by noise or artifacts).

False Negatives (Deceptive Clarity):
Highly structured or elegant works can score lower because they resolve quickly — even if they’re artistically profound. A 2.5 score doesn’t mean the image is “lesser”; it just means the viewer doesn’t need to work hard to parse it.

Overfitting to Complexity:
More complexity ≠ higher consequence. A crowded or chaotic image may overwhelm the viewer (high load) without ever producing meaningful recursion. The best high-VCLI images sustain focused ambiguity, not just confusion.

Structural vs. Perceptual Drift:
Remember: VCLI-G is blind to compositional correctness. A work with poor spatial logic might score high, while a beautifully structured image might score low. That’s why pairing VCLI with Sketcher (structure) or Artist Lens (poise and delay) gives a more complete picture.

✅ Rule of thumb:
A high VCLI-G score means the image demands attention — but only context, critique, and comparison will tell you whether that demand is earned or incidental. Use it as one dimension of analysis, not a verdict.

**Bonus Tables and Charts**

In [ ]:
# @title 1. VCLI-G / SCI
# Execute add-on (df must already exist from your base scoring cells)
m = run_sequence_metrics_addon(
    df,
    epsilon=0.05,                           # stability window tolerance
    write_json_path="sequence_metrics.json",  # optional
    append_ledger_path="admin_ledger.json",   # optional
    do_plots=True                           # two single-figure charts
)
m.to_dict()

2D Cognitive Space (VCLI-G × SCI).

Scatter: each row/iteration plotted in the VCLI-G (x) × SCI (y) plane.
Dashed lines are medians (robust splits); centroid labels the average state.
A dotted line shows the cloud’s principal orientation (major axis).

JSON summary: counts per quadrant (low/low, low/high, high/low, high/high), centroid/median, convex-hull area (spread of behaviors), and the cloud’s major-axis slope (tendency toward “more load with more coherence” vs “load rising as coherence falls”).

Top left (Variant 1) – efficiency pocket: low effort, high payoff.

Top right (Variant 3) – earned complexity: high effort, high payoff.

Bottom right (Variant 2) – strain pocket: high effort, lower payoff.

Bottom center (Variant 4) – underbuilt zone: middling effort, low payoff.

In [ ]:
# @title 12. VCLI-G x SCI
# === Execution cell: labeled VCLI-G × SCI scatter + printed summary (0910G-safe) ===
import numpy as np, pandas as pd, json
import matplotlib.pyplot as plt

def _convex_hull_area(x: np.ndarray, y: np.ndarray) -> float:
    pts = np.vstack([x, y]).T
    if pts.shape[0] < 3: return 0.0
    pts = pts[np.lexsort((pts[:,1], pts[:,0]))]
    def cross(o,a,b): return (a[0]-o[0])*(b[1]-o[1]) - (a[1]-o[1])*(b[0]-o[0])
    lower=[]
    for p in pts:
        while len(lower)>=2 and cross(lower[-2], lower[-1], p) <= 0: lower.pop()
        lower.append(tuple(p))
    upper=[]
    for p in pts[::-1]:
        while len(upper)>=2 and cross(upper[-2], upper[-1], p) <= 0: upper.pop()
        upper.append(tuple(p))
    hull = np.array(lower[:-1] + upper[:-1], dtype=float)
    if hull.shape[0] < 3: return 0.0
    xh, yh = hull[:,0], hull[:,1]
    return 0.5 * abs(np.dot(xh, np.roll(yh, -1)) - np.dot(yh, np.roll(xh, -1)))

def _principal_axis(x: np.ndarray, y: np.ndarray):
    if x.size < 2: return (np.array([0.0, 0.0]), np.nan)
    C = np.cov(np.vstack([x, y]))
    w, V = np.linalg.eig(C)
    i = np.argsort(w)[::-1]
    w = w[i]; V = V[:, i]
    vx, vy = V[0,0], V[1,0]
    slope = np.nan if abs(vx) < 1e-12 else (vy / vx)
    return w, slope

def plot_space_with_labels_and_summary(df: pd.DataFrame, title="VCLI-G × SCI (labeled)", write_json_path=None):
    # pull series
    V = pd.to_numeric(df["VCLI_G"], errors="coerce")
    S = pd.to_numeric(df["SCI"], errors="coerce")
    mask = ~(V.isna() | S.isna())
    Vc, Sc = V[mask].to_numpy(), S[mask].to_numpy()

    # labels: path → sequence_idx → row index
    if "path" in df.columns:
        labels = df.loc[mask, "path"].astype(str).to_numpy()
    elif "sequence_idx" in df.columns:
        labels = df.loc[mask, "sequence_idx"].astype(str).to_numpy()
    else:
        labels = np.array([str(i) for i in np.nonzero(mask)[0]])

    # robust splits (medians) + centroid
    mV, mS = float(np.nanmedian(Vc)), float(np.nanmedian(Sc))
    cV, cS = float(np.nanmean(Vc)),  float(np.nanmean(Sc))

    # spread/orientation
    hull_area = float(_convex_hull_area(Vc, Sc))
    eigvals, major_axis_slope = _principal_axis(Vc, Sc)

    # quadrant counts (by medians)
    q_ll = int(((Vc <  mV) & (Sc <  mS)).sum())
    q_lh = int(((Vc <  mV) & (Sc >= mS)).sum())
    q_hl = int(((Vc >= mV) & (Sc <  mS)).sum())
    q_hh = int(((Vc >= mV) & (Sc >= mS)).sum())

    summary = {
        "space_summary": {
            "n": int(Vc.size),
            "centroid": {"VCLI_G": cV, "SCI": cS},
            "median":   {"VCLI_G": mV, "SCI": mS},
            "hull_area": hull_area,
            "eigvals": [float(eigvals[0]), float(eigvals[1])],
            "major_axis_slope": None if np.isnan(major_axis_slope) else float(major_axis_slope),
            "quadrant_counts": {"low_low": q_ll, "low_high": q_lh, "high_low": q_hl, "high_high": q_hh}
        }
    }

    # plot
    plt.figure(figsize=(6.8, 5.6))
    plt.scatter(Vc, Sc)
    plt.axvline(mV, linestyle="--", alpha=0.6)
    plt.axhline(mS, linestyle="--", alpha=0.6)
    plt.scatter([cV], [cS])
    plt.text(cV, cS, "centroid", ha="left", va="bottom")
    for x, y, lab in zip(Vc, Sc, labels):
        plt.annotate(lab, (x, y), xytext=(3, 3), textcoords="offset points", fontsize=8)
    plt.xlabel("VCLI-G (cognitive load)")
    plt.ylabel("SCI (structural coherence)")
    plt.title(title)
    plt.tight_layout()
    plt.show()

    # print (and optionally save) summary
    if write_json_path:
        with open(write_json_path, "w", encoding="utf-8") as f:
            json.dump(summary, f, indent=2)
    print(summary)
    return summary

# run it
space = plot_space_with_labels_and_summary(df, write_json_path="vclig_sci_space.json")
# space  # already printed; uncomment to re-display the dict